# Chapter 9 — Ontologies

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 9. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
# TBox: inclusions A ⊑ B, definitions C ≡ A ⊓ B, disjointness
SUBCLASSES = {
    "MotorVehicle": ["Vehicle"],
    "Bicycle": ["Vehicle"],
    "Car": ["MotorVehicle"],
    "ElectricVehicle": ["MotorVehicle"],
    "Taxi": ["Car"],
    "ElectricTaxi": ["Taxi", "ElectricVehicle"],
}
DEFINITIONS = {"ElectricCar": ["Car", "ElectricVehicle"]}
DISJOINT = [("Bicycle", "MotorVehicle")]

# ABox: declared types of the individuals
INDIVIDUALS = {
    "vehicle_17": ["Car", "ElectricVehicle"],
    "taxi_42": ["ElectricTaxi"],
    "bike_9": ["Bicycle", "ElectricVehicle"],
}

def closure(types):
    """All implicit types: superclasses and satisfied defined classes."""
    types = set(types)
    changed = True
    while changed:
        changed = False
        for t in list(types):
            for parent in SUBCLASSES.get(t, []):
                if parent not in types:
                    types.add(parent)
                    changed = True
        for cls, conjuncts in DEFINITIONS.items():
            if cls not in types and all(c in types for c in conjuncts):
                types.add(cls)                # sufficient conditions
                changed = True
            if cls in types:
                for c in conjuncts:           # necessary conditions
                    if c not in types:
                        types.add(c)
                        changed = True
    return types

def classify():
    """For every named class, all its superclasses (subsumption)."""
    classes = (set(SUBCLASSES) | set(DEFINITIONS)
               | {s for parents in SUBCLASSES.values() for s in parents})
    return {c: sorted(closure([c]) - {c}) for c in classes}

def inconsistencies(types):
    return [(a, b) for a, b in DISJOINT if a in types and b in types]

hierarchy = classify()
print("ElectricTaxi ⊑", hierarchy["ElectricTaxi"])
print("ElectricCar ⊑", hierarchy["ElectricCar"])
for individual, declared in INDIVIDUALS.items():
    types = closure(declared)
    conflicts = inconsistencies(types)
    status = f"INCONSISTENT {conflicts}" if conflicts else "consistent"
    print(f"{individual}: ElectricCar={'ElectricCar' in types}, {status}")

## Expected output

If everything works, you should see:

```
ElectricTaxi ⊑ ['Car', 'ElectricCar', 'ElectricVehicle', 'MotorVehicle',
                'Taxi', 'Vehicle']
ElectricCar ⊑ ['Car', 'ElectricVehicle', 'MotorVehicle', 'Vehicle']
vehicle_17: ElectricCar=True, consistent
taxi_42: ElectricCar=True, consistent
bike_9: ElectricCar=False, INCONSISTENT [('Bicycle', 'MotorVehicle')]
```

Now try changing a number in the code and run it again to see what happens.